In [ ]:
# ============================================================
# ANN ADDITIONAL ANALYSIS: CV, ERROR DIAGNOSTICS, AND SENSITIVITY
# ============================================================
# This block is intended to be placed after the ANN regression
# models have been trained.
#
# Required objects from the ANN workflow:
# - df
# - X_COLS
# - FINAL_HIDDEN_LAYERS
# - FINAL_ALPHA
# - train/test predictions:
#     y_total_test, y_total_pred_test
#     y_leak_test, y_leak_pred_test
# - trained models and scalers:
#     mlp_total, scaler_X_total
#     mlp_leak, scaler_X_leak, scaler_y_leak
# - helper function:
#     manual_oversample_low_high
# - output folders:
#     FIGURES_DIR, TABLES_DIR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_error


# ============================================================
# SETTINGS
# ============================================================

RANDOM_STATE = 42
N_SPLITS = 5

TARGET_STORED = "mass_CO2_total"
TARGET_LEAKED = "mass_CO2_leakage"

pretty_feature_map = {
    "PERM_permeability": "Reservoir permeability",
    "PERM_perm_factor_of_failed_well": "Adjacent-well failure factor",
    "PERM_dist_to_well_in_grid": "Injector-to-well distance",
    "PERM_caprock_perm": "Caprock permeability",
    "porosity": "Reservoir porosity",
    "aquifer_porosity": "Aquifer porosity",
    "aquifer_permeability": "Aquifer permeability",
}


# ============================================================
# CROSS-VALIDATION ROBUSTNESS CHECKS
# ============================================================

df_cv = df[df["leak_category"] != "secure"].copy()
df_cv.reset_index(drop=True, inplace=True)

print("\n=== ANN CROSS-VALIDATION DATASET ===")
print("Dataset shape:", df_cv.shape)
print("\nClass distribution:")
print(df_cv["leak_category"].value_counts())

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

r2_scores_stored = []
mae_scores_stored = []

r2_scores_leaked = []
mae_scores_leaked = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(df_cv[X_COLS], df_cv["leak_category"]),
    start=1
):
    train_fold = df_cv.iloc[train_idx].copy()
    val_fold = df_cv.iloc[val_idx].copy()

    # Oversampling is applied only within the training fold.
    train_fold_os = manual_oversample_low_high(
        train_fold,
        label_col="leak_category",
        random_state=RANDOM_STATE
    )

    X_train_fold = train_fold_os[X_COLS].values
    X_val_fold = val_fold[X_COLS].values

    # --------------------------------------------------------
    # Stored CO2 mass model
    # --------------------------------------------------------

    y_train_stored = train_fold_os[TARGET_STORED].values
    y_val_stored = val_fold[TARGET_STORED].values

    model_stored = Pipeline([
        ("scaler_X", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=FINAL_HIDDEN_LAYERS,
            activation="relu",
            solver="adam",
            alpha=FINAL_ALPHA,
            learning_rate_init=0.001,
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])

    model_stored.fit(X_train_fold, y_train_stored)
    y_val_pred_stored = model_stored.predict(X_val_fold)

    r2_stored = r2_score(y_val_stored, y_val_pred_stored)
    mae_stored = mean_absolute_error(y_val_stored, y_val_pred_stored)

    r2_scores_stored.append(r2_stored)
    mae_scores_stored.append(mae_stored)

    # --------------------------------------------------------
    # Leaked CO2 mass model
    # --------------------------------------------------------

    y_train_leaked = train_fold_os[TARGET_LEAKED].values
    y_val_leaked = val_fold[TARGET_LEAKED].values

    base_model_leaked = Pipeline([
        ("scaler_X", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=FINAL_HIDDEN_LAYERS,
            activation="relu",
            solver="adam",
            alpha=FINAL_ALPHA,
            learning_rate_init=0.001,
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ])

    model_leaked = TransformedTargetRegressor(
        regressor=base_model_leaked,
        transformer=StandardScaler()
    )

    model_leaked.fit(X_train_fold, y_train_leaked)
    y_val_pred_leaked = model_leaked.predict(X_val_fold)

    y_val_pred_leaked = np.maximum(y_val_pred_leaked, 0.0)

    r2_leaked = r2_score(y_val_leaked, y_val_pred_leaked)
    mae_leaked = mean_absolute_error(y_val_leaked, y_val_pred_leaked)

    r2_scores_leaked.append(r2_leaked)
    mae_scores_leaked.append(mae_leaked)

    print(
        f"Fold {fold} | "
        f"Stored R² = {r2_stored:.4f}, Stored MAE = {mae_stored:.2f} t | "
        f"Leaked R² = {r2_leaked:.4f}, Leaked MAE = {mae_leaked:.2f} t"
    )

cv_results = pd.DataFrame({
    "Fold": np.arange(1, N_SPLITS + 1),
    "Stored_R2": r2_scores_stored,
    "Stored_MAE_t": mae_scores_stored,
    "Leaked_R2": r2_scores_leaked,
    "Leaked_MAE_t": mae_scores_leaked
})

cv_summary = pd.DataFrame({
    "Target": ["Stored CO2 mass", "Leaked CO2 mass"],
    "Mean_R2": [np.mean(r2_scores_stored), np.mean(r2_scores_leaked)],
    "Std_R2": [np.std(r2_scores_stored), np.std(r2_scores_leaked)],
    "Mean_MAE_t": [np.mean(mae_scores_stored), np.mean(mae_scores_leaked)],
    "Std_MAE_t": [np.std(mae_scores_stored), np.std(mae_scores_leaked)]
})

cv_results_display = cv_results.copy()
cv_results_display[["Stored_R2", "Leaked_R2"]] = (
    cv_results_display[["Stored_R2", "Leaked_R2"]].round(4)
)
cv_results_display[["Stored_MAE_t", "Leaked_MAE_t"]] = (
    cv_results_display[["Stored_MAE_t", "Leaked_MAE_t"]].round(2)
)

cv_summary_display = cv_summary.copy()
cv_summary_display[["Mean_R2", "Std_R2"]] = (
    cv_summary_display[["Mean_R2", "Std_R2"]].round(4)
)
cv_summary_display[["Mean_MAE_t", "Std_MAE_t"]] = (
    cv_summary_display[["Mean_MAE_t", "Std_MAE_t"]].round(2)
)

print("\n=== ANN CROSS-VALIDATION RESULTS BY FOLD ===")
print(cv_results_display.to_string(index=False))

print("\n=== ANN CROSS-VALIDATION SUMMARY ===")
print(cv_summary_display.to_string(index=False))

cv_results.to_csv(
    TABLES_DIR / "Appendix_A1_ANN_CrossValidation_byFold_exact.csv",
    index=False
)

cv_summary.to_csv(
    TABLES_DIR / "Appendix_A1_ANN_CrossValidation_summary_exact.csv",
    index=False
)

cv_results_display.to_csv(
    TABLES_DIR / "Appendix_A1_ANN_CrossValidation_byFold_rounded.csv",
    index=False
)

cv_summary_display.to_csv(
    TABLES_DIR / "Appendix_A1_ANN_CrossValidation_summary_rounded.csv",
    index=False
)


# ============================================================
# CROSS-VALIDATION FIGURES
# ============================================================

folds = np.arange(1, N_SPLITS + 1)
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5), dpi=600)

ax.bar(
    folds - width / 2,
    r2_scores_stored,
    width=width,
    color="#4A76FF",
    edgecolor="black",
    alpha=0.85,
    label="Stored CO₂ mass"
)

ax.bar(
    folds + width / 2,
    r2_scores_leaked,
    width=width,
    color="#FF6A6A",
    edgecolor="black",
    alpha=0.85,
    label="Leaked CO₂ mass"
)

ax.axhline(
    np.mean(r2_scores_stored),
    color="#4A76FF",
    linestyle="--",
    linewidth=1.8,
    label=f"Mean stored $R^2$ = {np.mean(r2_scores_stored):.4f}"
)

ax.axhline(
    np.mean(r2_scores_leaked),
    color="#FF6A6A",
    linestyle=":",
    linewidth=2.0,
    label=f"Mean leaked $R^2$ = {np.mean(r2_scores_leaked):.4f}"
)

ax.set_xticks(folds)
ax.set_xticklabels([f"Fold {i}" for i in folds])

ax.set_xlabel("Validation fold", fontsize=14)
ax.set_ylabel("$R^2$ score", fontsize=14)
ax.set_title("Five-Fold Cross-Validation Performance", fontsize=16, fontweight="bold")

ax.tick_params(axis="both", labelsize=12)
ax.grid(axis="y", alpha=0.25, linestyle="--")

legend = ax.legend(
    loc="lower right",
    frameon=True,
    fancybox=True,
    framealpha=0.90,
    edgecolor="black",
    fontsize=10
)

legend.get_frame().set_linewidth(0.8)
legend.get_frame().set_facecolor("white")

plt.tight_layout()

fig_path = FIGURES_DIR / "Appendix_A1_ANN_CrossValidation_R2.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"Saved: {fig_path}")


fig, ax = plt.subplots(figsize=(8, 5), dpi=600)

ax.bar(
    folds - width / 2,
    mae_scores_stored,
    width=width,
    color="#4A76FF",
    edgecolor="black",
    alpha=0.85,
    label="Stored CO₂ mass"
)

ax.bar(
    folds + width / 2,
    mae_scores_leaked,
    width=width,
    color="#FF6A6A",
    edgecolor="black",
    alpha=0.85,
    label="Leaked CO₂ mass"
)

ax.axhline(
    np.mean(mae_scores_stored),
    color="#4A76FF",
    linestyle="--",
    linewidth=1.8,
    label=f"Mean stored MAE = {np.mean(mae_scores_stored):.0f} t"
)

ax.axhline(
    np.mean(mae_scores_leaked),
    color="#FF6A6A",
    linestyle=":",
    linewidth=2.0,
    label=f"Mean leaked MAE = {np.mean(mae_scores_leaked):.0f} t"
)

ax.set_xticks(folds)
ax.set_xticklabels([f"Fold {i}" for i in folds])

ax.set_xlabel("Validation fold", fontsize=14)
ax.set_ylabel("MAE (t)", fontsize=14)
ax.set_title("Five-Fold Cross-Validation Error", fontsize=16, fontweight="bold")

ax.tick_params(axis="both", labelsize=12)
ax.grid(axis="y", alpha=0.25, linestyle="--")

legend = ax.legend(
    loc="upper right",
    frameon=True,
    fancybox=True,
    framealpha=0.90,
    edgecolor="black",
    fontsize=10
)

legend.get_frame().set_linewidth(0.8)
legend.get_frame().set_facecolor("white")

plt.tight_layout()

fig_path = FIGURES_DIR / "Appendix_A1_ANN_CrossValidation_MAE.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"Saved: {fig_path}")


# ============================================================
# ANN ABSOLUTE ERROR DIAGNOSTICS
# ============================================================

errors_stored = np.abs(y_total_test - y_total_pred_test)
errors_leaked = np.abs(y_leak_test - y_leak_pred_test)

error_summary = pd.DataFrame({
    "Target": ["Stored CO2 mass", "Leaked CO2 mass"],
    "Mean_absolute_error_t": [np.mean(errors_stored), np.mean(errors_leaked)],
    "Median_absolute_error_t": [np.median(errors_stored), np.median(errors_leaked)],
    "P90_absolute_error_t": [
        np.percentile(errors_stored, 90),
        np.percentile(errors_leaked, 90)
    ],
    "Max_absolute_error_t": [np.max(errors_stored), np.max(errors_leaked)]
})

error_summary_display = error_summary.copy()
error_cols = [
    "Mean_absolute_error_t",
    "Median_absolute_error_t",
    "P90_absolute_error_t",
    "Max_absolute_error_t"
]

error_summary_display[error_cols] = error_summary_display[error_cols].round(2)

print("\n=== ANN ABSOLUTE ERROR SUMMARY ===")
print(error_summary_display.to_string(index=False))

error_summary.to_csv(
    TABLES_DIR / "ANN_AbsoluteError_summary_exact.csv",
    index=False
)

error_summary_display.to_csv(
    TABLES_DIR / "ANN_AbsoluteError_summary_rounded.csv",
    index=False
)


# ============================================================
# ABSOLUTE ERROR DISTRIBUTION FIGURES
# ============================================================

def plot_absolute_error_histogram(errors, target_label, output_path):
    """
    Plot absolute error histogram for ANN test-set predictions.
    """
    fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

    ax.hist(
        errors,
        bins=15,
        alpha=0.85,
        edgecolor="black"
    )

    ax.axvline(
        np.mean(errors),
        linestyle="--",
        linewidth=2,
        label=f"Mean = {np.mean(errors):.0f} t"
    )

    ax.axvline(
        np.median(errors),
        linestyle=":",
        linewidth=2,
        label=f"Median = {np.median(errors):.0f} t"
    )

    ax.set_xlabel("Absolute error (t)", fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    ax.set_title(f"Absolute Error Distribution — {target_label}", fontsize=16)

    legend = ax.legend(
        loc="upper right",
        frameon=True,
        fancybox=True,
        framealpha=0.90,
        edgecolor="black",
        fontsize=11
    )

    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_facecolor("white")

    ax.grid(alpha=0.25)

    plt.tight_layout()
    fig.savefig(output_path, dpi=600, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")


plot_absolute_error_histogram(
    errors=errors_stored,
    target_label="Stored CO₂ Mass",
    output_path=FIGURES_DIR / "Figure_9a_ANN_AbsoluteError_StoredCO2Mass.png"
)

plot_absolute_error_histogram(
    errors=errors_leaked,
    target_label="Leaked CO₂ Mass",
    output_path=FIGURES_DIR / "Figure_9b_ANN_AbsoluteError_LeakedCO2Mass.png"
)


# ============================================================
# MORRIS-LIKE SENSITIVITY SCREENING
# ============================================================

def predict_stored_mass_physical(X_raw):
    """
    Predict stored CO2 mass in physical units.
    """
    X_scaled = scaler_X_total.transform(X_raw)
    return mlp_total.predict(X_scaled)


def predict_leaked_mass_physical(X_raw):
    """
    Predict leaked CO2 mass in physical units.

    The leaked-mass ANN was trained using a standardized target,
    so predictions are inverse-transformed.
    """
    X_scaled = scaler_X_leak.transform(X_raw)
    y_pred_scaled = mlp_leak.predict(X_scaled)
    y_pred = scaler_y_leak.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    return np.maximum(y_pred, 0.0)


def morris_like_normalized_sensitivity(
    model_predict_function,
    X_base_raw,
    scaler_X,
    input_columns,
    delta=0.1
):
    """
    Compute normalized one-at-a-time sensitivity values.

    Each standardized input feature is perturbed by a fixed increment
    while the remaining features are held constant. Sensitivities are
    computed as the mean absolute prediction change in physical units
    and normalized by the maximum effect for each target variable.
    """
    X_base_raw = np.asarray(X_base_raw).copy()

    base_pred = model_predict_function(X_base_raw)
    X_base_scaled = scaler_X.transform(X_base_raw)

    raw_effects = []

    for i, _ in enumerate(input_columns):
        X_perturbed_scaled = X_base_scaled.copy()
        X_perturbed_scaled[:, i] += delta

        X_perturbed_raw = scaler_X.inverse_transform(X_perturbed_scaled)

        perturbed_pred = model_predict_function(X_perturbed_raw)

        effect = np.mean(np.abs(perturbed_pred - base_pred))
        raw_effects.append(effect)

    raw_effects = np.array(raw_effects)

    if raw_effects.max() > 0:
        normalized_effects = raw_effects / raw_effects.max()
    else:
        normalized_effects = raw_effects

    return normalized_effects, raw_effects


df_sens = df[df["leak_category"] != "secure"].copy()
df_sens.reset_index(drop=True, inplace=True)

X_sens = df_sens[X_COLS].values

print("\n=== MORRIS-LIKE SENSITIVITY BASE DATASET ===")
print("Dataset shape:", X_sens.shape)
print(df_sens["leak_category"].value_counts())

stored_sens_norm, stored_sens_raw = morris_like_normalized_sensitivity(
    model_predict_function=predict_stored_mass_physical,
    X_base_raw=X_sens,
    scaler_X=scaler_X_total,
    input_columns=X_COLS,
    delta=0.1
)

leaked_sens_norm, leaked_sens_raw = morris_like_normalized_sensitivity(
    model_predict_function=predict_leaked_mass_physical,
    X_base_raw=X_sens,
    scaler_X=scaler_X_leak,
    input_columns=X_COLS,
    delta=0.1
)

pretty_features = [pretty_feature_map.get(col, col) for col in X_COLS]

morris_table = pd.DataFrame({
    "Input variable": pretty_features,
    "Stored CO2 mass": stored_sens_norm,
    "Leaked CO2 mass": leaked_sens_norm,
    "Stored raw effect (t)": stored_sens_raw,
    "Leaked raw effect (t)": leaked_sens_raw
})

morris_table_sorted = morris_table.sort_values(
    by="Leaked CO2 mass",
    ascending=False
).reset_index(drop=True)

morris_table_display = morris_table_sorted.copy()

morris_table_display[["Stored CO2 mass", "Leaked CO2 mass"]] = (
    morris_table_display[["Stored CO2 mass", "Leaked CO2 mass"]].round(2)
)

morris_table_display[["Stored raw effect (t)", "Leaked raw effect (t)"]] = (
    morris_table_display[["Stored raw effect (t)", "Leaked raw effect (t)"]].round(2)
)

print("\n=== TABLE 8 — MORRIS-LIKE NORMALIZED SENSITIVITY ===")
print(morris_table_display.to_string(index=False))

morris_table_sorted.to_csv(
    TABLES_DIR / "Table_8_MorrisLikeSensitivity_exact.csv",
    index=False
)

morris_table_display.to_csv(
    TABLES_DIR / "Table_8_MorrisLikeSensitivity_rounded.csv",
    index=False
)

print("\nSaved:")
print(f"- {TABLES_DIR / 'Table_8_MorrisLikeSensitivity_exact.csv'}")
print(f"- {TABLES_DIR / 'Table_8_MorrisLikeSensitivity_rounded.csv'}")